# Advanced Time Series Analysis in Machine Learning

This notebook focuses on advanced, practical ML workflows for time series forecasting using a synthetic retail demand dataset.

## What you will learn
- Time-aware train/validation strategy (`TimeSeriesSplit`)
- Lag features and rolling-window feature engineering
- Recursive vs direct multi-step forecasting
- Advanced regressors for forecasting:
  - Random Forest Regressor
  - HistGradientBoosting Regressor
  - Support Vector Regressor (with scaling)
- Walk-forward style evaluation and model comparison
- Forecast diagnostics and error analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1) Build a realistic time series

We simulate daily demand with trend, weekly seasonality, yearly seasonality, and random noise.

In [ ]:
n_days = 3 * 365
dates = pd.date_range(start='2021-01-01', periods=n_days, freq='D')
t = np.arange(n_days)

trend = 0.01 * t
weekly = 6 * np.sin(2 * np.pi * t / 7)
yearly = 12 * np.sin(2 * np.pi * t / 365)
noise = np.random.normal(0, 2.5, size=n_days)

y = 40 + trend + weekly + yearly + noise

df = pd.DataFrame({'date': dates, 'y': y}).set_index('date')
df.head()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df.index, df['y'], lw=1.2)
plt.title('Daily Demand Time Series')
plt.xlabel('Date')
plt.ylabel('Demand')
plt.tight_layout()
plt.show()

## 2) Feature engineering for time series ML

In [ ]:
def make_features(series: pd.Series) -> pd.DataFrame:
    x = pd.DataFrame(index=series.index)
    x['y'] = series

    # Lag features
    for lag in [1, 2, 3, 7, 14, 28]:
        x[f'lag_{lag}'] = series.shift(lag)

    # Rolling statistics (using past values only)
    for w in [7, 14, 28]:
        x[f'roll_mean_{w}'] = series.shift(1).rolling(w).mean()
        x[f'roll_std_{w}'] = series.shift(1).rolling(w).std()

    # Calendar features
    x['dayofweek'] = x.index.dayofweek
    x['month'] = x.index.month
    x['dayofyear'] = x.index.dayofyear

    # Cyclical encoding
    x['dow_sin'] = np.sin(2 * np.pi * x['dayofweek'] / 7)
    x['dow_cos'] = np.cos(2 * np.pi * x['dayofweek'] / 7)
    x['m_sin'] = np.sin(2 * np.pi * x['month'] / 12)
    x['m_cos'] = np.cos(2 * np.pi * x['month'] / 12)

    return x.dropna()

feat_df = make_features(df['y'])
feat_df.head()

## 3) Time-aware train/test split

Important: never shuffle time series.

In [ ]:
target = 'y'
features = [c for c in feat_df.columns if c != target]

split_idx = int(len(feat_df) * 0.8)
train_df = feat_df.iloc[:split_idx]
test_df = feat_df.iloc[split_idx:]

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)

## 4) Advanced model set

In [ ]:
models = {
    'RandomForest': RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    'HistGradientBoosting': HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_iter=500,
        random_state=RANDOM_STATE,
    ),
    'SVR': Pipeline([
        ('scaler', StandardScaler()),
        ('svr', SVR(C=10, epsilon=0.2, kernel='rbf')),
    ]),
}

models

## 5) Cross-validation with TimeSeriesSplit

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
scoring = {
    'mae': 'neg_mean_absolute_error',
    'rmse': 'neg_root_mean_squared_error',
}

rows = []
for name, model in models.items():
    cv = cross_validate(model, X_train, y_train, cv=tscv, scoring=scoring)
    rows.append({
        'model': name,
        'cv_mae': -cv['test_mae'].mean(),
        'cv_rmse': -cv['test_rmse'].mean(),
    })

cv_results = pd.DataFrame(rows).sort_values('cv_rmse')
cv_results

## 6) Fit best model and evaluate on holdout

In [ ]:
best_name = cv_results.iloc[0]['model']
best_model = models[best_name]

best_model.fit(X_train, y_train)
pred_test = best_model.predict(X_test)

mae = mean_absolute_error(y_test, pred_test)
rmse = np.sqrt(mean_squared_error(y_test, pred_test))

print('Best model:', best_name)
print(f'Test MAE:  {mae:.3f}')
print(f'Test RMSE: {rmse:.3f}')

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(y_train.index[-120:], y_train.values[-120:], label='Train (last 120)', lw=1)
plt.plot(y_test.index, y_test.values, label='Actual (test)', lw=2)
plt.plot(y_test.index, pred_test, label=f'Predicted ({best_name})', lw=2)
plt.title('Holdout Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Demand')
plt.legend()
plt.tight_layout()
plt.show()

## 7) Multi-step forecasting strategies

### Recursive strategy
Predict one step ahead repeatedly, feeding previous predictions back as lags.

In [ ]:
def recursive_forecast(last_history: pd.Series, horizon: int, model, full_index: pd.DatetimeIndex):
    hist = last_history.copy()
    preds = []

    for i in range(horizon):
        current_idx = full_index[i]
        tmp_series = pd.concat([hist, pd.Series([np.nan], index=[current_idx])])
        tmp_feat = make_features(tmp_series).iloc[-1:]

        x_row = tmp_feat[[c for c in tmp_feat.columns if c != 'y']]
        y_hat = model.predict(x_row)[0]

        preds.append(y_hat)
        hist.loc[current_idx] = y_hat

    return np.array(preds)

future_horizon = 30
future_idx = pd.date_range(start=df.index[-1] + pd.Timedelta(days=1), periods=future_horizon, freq='D')
future_pred = recursive_forecast(df['y'], future_horizon, best_model, future_idx)

forecast_df = pd.DataFrame({'forecast': future_pred}, index=future_idx)
forecast_df.head()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df.index[-120:], df['y'].iloc[-120:], label='Recent history', lw=1.5)
plt.plot(forecast_df.index, forecast_df['forecast'], label='30-day recursive forecast', lw=2)
plt.title('Future Forecast (Recursive Strategy)')
plt.xlabel('Date')
plt.ylabel('Demand')
plt.legend()
plt.tight_layout()
plt.show()

## 8) Feature importance (tree-based model)

In [ ]:
if best_name == 'RandomForest':
    importances = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)

    plt.figure(figsize=(8, 6))
    importances.tail(15).plot(kind='barh')
    plt.title('Top 15 Feature Importances (Random Forest)')
    plt.tight_layout()
    plt.show()
else:
    print('Feature importances shown only when RandomForest is the best model.')

## 9) Extensions you can add
- Probabilistic forecasting (prediction intervals)
- Global forecasting with panel data (multiple time series)
- Drift detection and automatic model retraining
- Direct multi-horizon forecasting with `MultiOutputRegressor`
- Compare with classical methods (ARIMA/SARIMA) in a separate notebook